In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np

In [19]:
from bayesgpt.simulators import ModelVariant, Tokenizer
from bayesgpt.simulators.benchmarks import SuperDDM, StandardDDM, CollapsingBoundDDM

### Metas

In [20]:
num_samples = 1000  # Global number of samples per model variant

In [21]:
# Define modulation function for context-dependent parameters in SuperDDM
def modulation(params, context):
    """Adjust drift rate based on context (e.g., stimulus strength)."""
    params = params.copy()
    if "v" in params:
        params["v"] = params["v"] * (1 + context[0])  # Scale drift rate
    elif "v_components" in params:
        params["v_components"] = params["v_components"] * (1 + context[0])
    elif "v_schedule" in params:
        params["v_schedule"] = params["v_schedule"] * (1 + context[0])
    return params

In [22]:
# Common tokenizer parameters
parameter_names = [
    "v",
    "a",
    "z",
    "tau",
    "sigma",
    "angle",
    "s_v",
    "s_z",
    "s_tau",
    "v_components",
    "p_components",
    "v_schedule",
    "t_schedule"
]

### DDM Variants

In [8]:
super_ddm_params = parameter_names

In [13]:
fixed_parameters = {
    "p_components": np.array([0.6, 0.4]),  # Mixture probabilities
    "t_schedule": np.array([0.0, 0.5])  # Time points for scheduled drifts
}
free_parameters = {
    "a": lambda c: np.random.uniform(0.8, 1.2, 1),  # Decision boundary
    "sigma": lambda c: np.random.uniform(0.05, 0.15, 1),  # Diffusion noise
    "s_v": lambda c: np.random.uniform(0.01, 0.1, 1),  # Drift rate noise
    "angle": lambda c: np.random.uniform(0.0, 0.05, 1),  # Boundary collapse
    "s_z": lambda c: np.random.uniform(0.005, 0.02, 1),  # Starting point noise
    "s_tau": lambda c: np.random.uniform(0.005, 0.02, 1),  # Non-decision time noise
    "v_components": lambda c: np.random.randn(2) * 0.5,  # Mixture drift rates
    "v_schedule": lambda c: np.random.randn(2) * 0.5,  # Scheduled drift rates
    "z": lambda c: np.random.uniform(0.4, 0.6, 1),  # Starting point
    "tau": lambda c: np.random.uniform(0.1, 0.3, 1)  # Non-decision time
}
parameter_dims = {
    "a": 1, "v": 1, "sigma": 1, "s_v": 1, "z": 1, "tau": 1, "angle": 1,
    "s_z": 1, "s_tau": 1, "v_components": 2, "p_components": 2,
    "v_schedule": 2, "t_schedule": 2
}

In [23]:
# Initialize Tokenizers with different variant_parameters
tokenizer_super_mixture = Tokenizer(
    parameter_names=parameter_names,
    variant_parameters=["a", "v_components", "p_components", "sigma", "s_v", "z", "tau", "angle", "s_z", "s_tau"],
    fixed_parameters=fixed_parameters,
    free_parameters=free_parameters,
    parameter_dims=parameter_dims,
    context_shape=(1,)
)
tokenizer_super_schedule = Tokenizer(
    parameter_names=parameter_names,
    variant_parameters=["a", "v_schedule", "t_schedule", "sigma", "s_v", "z", "tau", "angle", "s_z", "s_tau"],
    fixed_parameters=fixed_parameters,
    free_parameters=free_parameters,
    parameter_dims=parameter_dims,
    context_shape=(1,)
)
tokenizer_standard = Tokenizer(
    parameter_names=parameter_names,
    variant_parameters=["v", "a", "sigma", "s_v", "z", "tau", "s_z", "s_tau"],  # No angle
    fixed_parameters=fixed_parameters,
    free_parameters=free_parameters,
    parameter_dims=parameter_dims,
    context_shape=(1,)
)
tokenizer_collapsing = Tokenizer(
    parameter_names=parameter_names,
    variant_parameters=["v", "a", "sigma", "s_v", "z", "tau", "angle", "s_z", "s_tau"],
    fixed_parameters=fixed_parameters,
    free_parameters=free_parameters,
    parameter_dims=parameter_dims,
    context_shape=(1,)
)

In [24]:
model_variant_mixture = ModelVariant(
    name="super_ddm_mixture",
    model=SuperDDM,
    tokenizer=tokenizer_super_mixture,
    num_samples=num_samples
)
model_variant_schedule = ModelVariant(
    name="super_ddm_schedule",
    model=SuperDDM,
    tokenizer=tokenizer_super_schedule,
    num_samples=num_samples
)
model_variant_standard = ModelVariant(
    name="standard_ddm",
    model=StandardDDM,
    tokenizer=tokenizer_standard,
    num_samples=num_samples
)
model_variant_collapsing = ModelVariant(
    name="collapsing_bound_ddm",
    model=CollapsingBoundDDM,
    tokenizer=tokenizer_collapsing,
    num_samples=num_samples
)

In [25]:
# Cell 3: Run simulations and print results
context = np.array([0.5], dtype=np.float32)  # Single simulation context
result_super_mixture = model_variant_mixture.sample(context=context)
result_super_schedule = model_variant_schedule.sample(context=context)
result_standard = model_variant_standard.sample(context=context)
result_collapsing = model_variant_collapsing.sample(context=context)

ValueError: Missing parameters: {'angle'}

In [18]:
# Summarize mixture model
summary_mixture = SuperDDM.summarize(
    outputs=result_mixture["sim_data"],
    quantile_levels=[0.1, 0.3, 0.5, 0.7, 0.9],
    by_choice=True,
    tau=np.full(num_samples, result_mixture["full_params"][
        super_ddm_tokenizer.parameter_slices["tau"]][0], dtype=np.float32)
)
print("Mixture Model Summary:")
print("Invalid Rate:", summary_mixture["invalid_rate"])
print("RT Quantiles:", summary_mixture["rt_quantiles"])
print("RT Quantiles by Choice:\n", summary_mixture["rt_quantiles_by_choice"])
print("Decision Time Quantiles:", summary_mixture["dt_quantiles"])
print("Decision Time Quantiles by Choice:\n", summary_mixture["dt_quantiles_by_choice"])

# Summarize scheduled model
summary_schedule = SuperDDM.summarize(
    outputs=result_schedule["sim_data"],
    quantile_levels=[0.1, 0.3, 0.5, 0.7, 0.9],
    by_choice=True,
    tau=np.full(num_samples, result_schedule["full_params"][
        super_ddm_tokenizer.parameter_slices["tau"]][0], dtype=np.float32)
)
print("\nScheduled Model Summary:")
print("Invalid Rate:", summary_schedule["invalid_rate"])
print("RT Quantiles:", summary_schedule["rt_quantiles"])
print("RT Quantiles by Choice:\n", summary_schedule["rt_quantiles_by_choice"])
print("Decision Time Quantiles:", summary_schedule["dt_quantiles"])
print("Decision Time Quantiles by Choice:\n", summary_schedule["dt_quantiles_by_choice"])

Mixture Model Summary:
Invalid Rate: 0.007
RT Quantiles: [2.886325 3.431478 3.842053 4.506588 5.637077]
RT Quantiles by Choice:
 [[2.8872917 3.4327617 3.8435135 4.5078607 5.637225 ]
 [2.015695  2.015695  2.015695  2.015695  2.015695 ]]
Decision Time Quantiles: [2.5929446 3.1380978 3.5486727 4.2132077 5.3436966]
Decision Time Quantiles by Choice:
 [[2.5939114 3.1393814 3.5501332 4.2144804 5.343845 ]
 [1.7223148 1.7223148 1.7223148 1.7223148 1.7223148]]

Scheduled Model Summary:
Invalid Rate: 0.0
RT Quantiles: [3.3494363 3.748377  4.0990686 4.51606   5.161834 ]
RT Quantiles by Choice:
 [[3.3494363 3.748377  4.0990686 4.51606   5.161834 ]
 [      nan       nan       nan       nan       nan]]
Decision Time Quantiles: [3.1262887 3.5252295 3.8759212 4.292912  4.938686 ]
Decision Time Quantiles by Choice:
 [[3.1262887 3.5252295 3.8759212 4.292912  4.938686 ]
 [      nan       nan       nan       nan       nan]]


### Model Variants

You can use `ModelVariant` directly.

In [6]:
# Variant 1: Single drift
tokenizer_single = Tokenizer(
    parameter_names=parameter_names,
    variant_parameters={"v", "a", "z", "tau", "sigma", "angle", "s_v", "s_z", "s_tau"},
    fixed_parameters=fixed_parameters,
    free_parameters={"v": lambda c: np.random.normal(0.5, 0.1, 1)},
    parameter_dims=parameter_dims,
    context_shape=(1,)
)
variant_single = ModelVariant("single_drift", SuperDDM, tokenizer_single, num_samples)
result_single = variant_single.sample(context=context)

ValueError: a must have shape (num_samples,), but got shape (1,)

In [5]:
model_family = NestedModelFamily(parameter_names=param_names)

In [16]:
model_family.sample("standard", batch_size=3)

ValueError: All parameters must be scalars or arrays of shape (num_trials,)